# Semantic voting for Qiskit code generation
**A test-time-scaling mini-project on Qiskit-HumanEval (Hugging Face: `Qiskit/qiskit_humaneval`)**

**Question.** If you sample N programs from a small code LLM and have *no unit tests*, can you pick a good one
by running them on a simulator and keeping the answer most candidates agree on (the biggest cluster of
*semantically equivalent* circuits)? And can you stop sampling early once the clusters settle?

**What this notebook does**
1. Loads the 151 Qiskit-HumanEval tasks and checks which canonical solutions still work in your Qiskit version.
2. Samples N completions per task from a small open model.
3. Executes every completion, fingerprints its output (circuit -> unitary, counts -> distribution, ...), clusters equivalent ones.
4. Compares selection strategies as N grows: `random`, `valid_filter` (just "does it run?"), `semantic` (biggest cluster), `oracle` (pass@N, i.e. what perfect tests would give).
5. Tests an adaptive stopping rule (stop sampling once one cluster clearly leads) and plots accuracy vs. tokens.
6. **(v2) Statistics for a paper:** exact `random`/`oracle`, bootstrap confidence intervals over tasks, paired differences, a failure-type breakdown (Step 10).
7. **(v2) Baselines and ablations:** syntactic majority vote, equivalence-definition ablation, no-argument-task subset (Step 11).
8. **(v2) Adaptive stopping done properly:** Adaptive-Consistency's Beta rule, threshold chosen on held-out tasks, comparison with fixed N at matched budgets (Step 12).
9. **(v2, GPU) Greedy-decoding baseline** (Step 13) and, optionally, **fingerprint inputs written by the LLM** instead of taken from the tests (Step 14).
10. **(v2.1) Object-reading fingerprint:** circuit DAGs, scalar operators, instructions and noise models are compared by content instead of by type name only (`OPEN_OBJECTS` in the harness cell; Step 5a refreshes the cache).

**Honest assumptions (put these in your paper's Method/Limitations):**
* Selection never uses the tests' *asserts or expected values*. It does reuse the **call inputs** that the test makes (recorded by running the canonical solution once). In deployment those inputs would come from the engineer or from an LLM. Step 11 reports the tasks that need no inputs at all, and Step 14 (optional) replaces the test inputs by LLM-written ones.
* Candidates that crash, time out, or return `None` are excluded from voting (label-free sanity filter).
* The set of *usable* tasks (canonical solution passes in your environment) and the *informative-fingerprint* subset are both chosen using the canonical solution. Report both, and report how many tasks were excluded and why (Step 15).
* Unitary comparison is only done for circuits with at most 8 qubits and no free parameters; larger or parametrised circuits fall back to a structural fingerprint. Other objects are compared by type name only, except the few types that v2.1 opens.
* The benchmark's tests are *property checks* (e.g. "counts has keys 00 and 11"), not "equals the reference", so a cluster can contain both passing and failing programs. We report the expected accuracy of picking a random member of the winning cluster.

**v3** (built from v2.1). Changes: pinned install (cell 1); reproducible fingerprinting that keeps programs' own seeds and leaves the benchmark's tests unseeded (cell 8); signature check for LLM-written example calls (Step 14); atomic cache saves; extra analysis cells (Table 3, Table 9, no-argument split); optional re-validation and determinism cells.

**For each new run change only** `MODEL_ID`, `SEED` and `OUT_DIR` in the config cell (keep `RUN_LLM_INPUTS = True`). Run cell 1 once, restart the session, then continue from the environment check. Cell 10 must report 138 usable tasks.


In [ ]:
# v3: exact versions used for every reported run (symengine must stay below 0.14 for Qiskit 1.4.6).
# After running this cell once: Runtime -> Restart session, then continue with the next cell.
!pip -q install "qiskit==1.4.6" "qiskit-aer==0.18.2" "qiskit-ibm-runtime==0.43.1" "symengine>=0.11,<0.14" pylatexenc matplotlib requests "transformers>=4.44" accelerate

### Environment check
Run this once after the install cell **and a session restart**. It should print three versions and `plugin module imports cleanly`. An `ImportError` that mentions `calc_final_ops` means `qiskit-ibm-runtime` is too new for Qiskit 1.x.

In [ ]:
import importlib.metadata as md
print({p: md.version(p) for p in ["qiskit", "qiskit-aer", "qiskit-ibm-runtime"]})
import qiskit_ibm_runtime.transpiler.plugin   # ImportError here = Qiskit / qiskit-ibm-runtime versions still mismatched
print("plugin module imports cleanly")

In [ ]:
# v3: for a new run change only MODEL_ID, SEED, OUT_DIR (and keep RUN_LLM_INPUTS = True). Everything else stays fixed.
import os
os.environ["OMP_NUM_THREADS"] = "1"   # avoid thread oversubscription in worker processes
import re, io, sys, json, time, copy, math, random, pickle, warnings, contextlib, functools, gc
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# ------------------------------ CONFIG ------------------------------
MODEL_ID       = "Qwen/Qwen2.5-Coder-1.5B"   # base code model; fits a free Colab T4
N_SAMPLES      = 32                          # completions per task
TEMPERATURE    = 0.7
TOP_P          = 0.95
MAX_NEW_TOKENS = 384
MAX_TASKS      = None      # set e.g. 10 (and N_SAMPLES = 8) for a quick smoke test
EXEC_TIMEOUT   = 20        # seconds per candidate program
N_WORKERS      = 2         # Colab free tier has 2 CPU cores
SEED           = 0
OUT_DIR        = "qhe_run"
# How to decide that two *circuits* are "the same answer":
#   "unitary" = same unitary up to global phase (strict; right for gate/algorithm tasks)
#   "state"   = same state when started from |0...0> (looser; right for state-preparation tasks)
# Execution results are cached, so you can change this and re-run from Step 5 to get an ablation.
CIRCUIT_EQ     = "unitary"
# --- v2 additions (paper-grade analysis) ---
BOOT_B         = 2000      # bootstrap resamples over tasks for confidence intervals
RUN_GREEDY     = True      # Step 13: greedy-decoding baseline (GPU, ~138 short generations)
RUN_LLM_INPUTS = True      # Step 14: fingerprint inputs written by the LLM instead of taken from the tests (GPU + ~1-2 h CPU)
N_LLM_INPUTS   = 3         # example calls per task for Step 14
# Optional: keep results across Colab disconnects
# from google.colab import drive; drive.mount("/content/drive"); OUT_DIR = "/content/drive/MyDrive/qhe_run"
os.makedirs(OUT_DIR, exist_ok=True)

## Step 1: load the dataset from Hugging Face

In [ ]:
import requests

def load_qhe_rows(dataset="Qiskit/qiskit_humaneval", config="default", split="test"):
    """Download the dataset through Hugging Face's JSON API, so no `datasets` / pyarrow install is needed."""
    rows, offset = [], 0
    while True:
        r = requests.get("https://datasets-server.huggingface.co/rows", timeout=60,
                         params=dict(dataset=dataset, config=config, split=split, offset=offset, length=100))
        r.raise_for_status()
        j = r.json()
        batch = j["rows"]
        if any(b.get("truncated_cells") for b in batch):
            raise ValueError("the API truncated some cells")
        rows += [b["row"] for b in batch]
        offset += len(batch)
        if not batch or offset >= j["num_rows_total"]:
            return rows

try:
    tasks = load_qhe_rows()
except Exception as e:
    print(f"JSON API failed ({e!r}); falling back to the `datasets` library (needs: pip install datasets)")
    from datasets import load_dataset
    tasks = [dict(r) for r in load_dataset("Qiskit/qiskit_humaneval", split="test")]

need = {"task_id", "prompt", "canonical_solution", "test", "entry_point", "difficulty_scale"}
assert tasks and need <= set(tasks[0]), f"unexpected dataset format: {list(tasks[0]) if tasks else tasks}"
print(len(tasks), "tasks; fields:", list(tasks[0].keys()))
print(Counter(t["difficulty_scale"] for t in tasks))
t0 = tasks[0]
print("\n--- prompt ---\n" + t0["prompt"])
print("--- test ---\n" + t0["test"])

## Step 2: the execution harness
* Every program runs in its own **forked process with a hard timeout** (LLM code can hang, e.g. by calling real hardware).
* `make_sig` turns whatever a function returns into a **fingerprint**: circuits become unitaries (up to global phase, final measurements stripped), `Statevector`/`Operator` are compared directly, counts dicts become probability distributions, numbers/arrays/strings are compared by value.
* `sig_equal` decides whether two fingerprints are "the same answer".

In [ ]:
import multiprocessing as mp
from multiprocessing.connection import wait as mp_wait
ctx = mp.get_context("fork")
# Pre-import heavy libraries ONCE here, so every forked worker starts instantly instead of re-importing Qiskit.
import qiskit, qiskit_aer, qiskit.quantum_info, qiskit.compiler, qiskit.circuit.library
for _m in ["qiskit.primitives", "qiskit.visualization", "qiskit_ibm_runtime", "qiskit_ibm_runtime.fake_provider"]:
    try: __import__(_m)
    except Exception: pass
DIST_TOL = 0.1   # max total-variation distance for two counts distributions to count as equal
OPEN_OBJECTS = True   # v2.1: also compare circuit DAGs, scalar operators, instructions and noise models by CONTENT (False = by type name only, as in v2)

def build_program(prompt, completion):
    if prompt and not prompt.endswith("\n") and not completion.startswith("\n"):
        prompt += "\n"
    return prompt + completion + "\n"

# ---------- fingerprints ----------
def circuit_sig(qc):
    from qiskit.quantum_info import Operator
    n_meas = qc.count_ops().get("measure", 0)
    try:
        if qc.num_qubits <= 8 and len(qc.parameters) == 0:
            U = Operator(qc.remove_final_measurements(inplace=False)).data
            return ("unitary", np.asarray(U), n_meas)
    except Exception:
        pass
    return ("struct", (qc.num_qubits, qc.num_clbits, tuple(sorted(qc.count_ops().items()))))

import re

# ---------- v2.1: read the CONTENT of a few object types that tests inspect (instead of only their type name) ----------
def _plain(x, depth=0):
    """Reduce nested dicts / lists / numbers / strings (for example NoiseModel.to_dict()) to a plain structure that can be compared."""
    if depth > 8: return ("deep",)
    if x is None or isinstance(x, str): return x
    if isinstance(x, (bool, np.bool_)): return bool(x)
    if isinstance(x, (int, float, np.integer, np.floating)): return float(x)
    if isinstance(x, (complex, np.complexfloating)): return ("c", float(x.real), float(x.imag))
    if isinstance(x, np.ndarray): return _plain(x.tolist(), depth + 1)
    if isinstance(x, dict):
        return ("d", tuple((str(k), _plain(v, depth + 1)) for k, v in sorted(x.items(), key=lambda kv: str(kv[0]))
                           if str(k) not in ("id", "uuid")))       # random ids would make equal objects look different
    if isinstance(x, (list, tuple)): return ("l", tuple(_plain(v, depth + 1) for v in x))
    return ("o", type(x).__name__)      # unknown leaf: only its type name (a repr would contain memory addresses)

def _plain_equal(a, b):
    if isinstance(a, float) and isinstance(b, float): return bool(np.isclose(a, b, rtol=1e-3, atol=1e-6))
    if type(a) is not type(b): return False
    if isinstance(a, tuple): return len(a) == len(b) and all(_plain_equal(x, y) for x, y in zip(a, b))
    return a == b

_OPENABLE_DICT_TYPES = {"NoiseModel", "QuantumError", "ReadoutError"}   # objects whose to_dict() describes their content deterministically

def _open_object(o):
    """Content-based signature for the object types that the benchmark's tests look inside. None = not one of them."""
    names = {c.__name__ for c in type(o).__mro__}
    try:
        if "DAGCircuit" in names:                       # tests check qubits, depth and gate counts
            return ("dag", (int(o.num_qubits()), int(o.num_clbits()), int(o.depth()),
                            tuple(sorted((str(k), int(v)) for k, v in dict(o.count_ops()).items()))))
        if "ScalarOp" in names:                         # tests check the coefficient and the dimensions
            c = complex(o.coeff)
            return ("scalarop", (c.real, c.imag, tuple(int(d) for d in o.input_dims()), tuple(int(d) for d in o.output_dims())))
        if "Instruction" in names and hasattr(o, "name") and hasattr(o, "params"):     # tests check the operation name
            name = re.sub(r"-\d+$", "-N", str(o.name))  # auto-generated names such as 'circuit-123' differ between programs
            return ("instr", (name, int(o.num_qubits), int(o.num_clbits), _plain(list(o.params))))
        if names & _OPENABLE_DICT_TYPES and callable(getattr(o, "to_dict", None)):     # noise models: tests check the error probabilities
            d = o.to_dict()
            if isinstance(d, (dict, list)): return ("plain", _plain(d))
    except Exception:
        return None
    return None

def make_sig(o, depth=0):
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector, Operator, DensityMatrix
    if depth > 3: return ("opaque", type(o).__name__)
    if o is None: return ("none",)
    if isinstance(o, (bool, np.bool_, int, float, np.integer, np.floating)): return ("num", float(o))
    if isinstance(o, (complex, np.complexfloating)): return ("array", np.array([o]))
    if isinstance(o, str): return ("str", o)
    if isinstance(o, QuantumCircuit): return circuit_sig(o)
    if isinstance(o, Statevector): return ("state", np.asarray(o.data))
    if isinstance(o, DensityMatrix): return ("array", np.asarray(o.data))
    if isinstance(o, Operator): return ("unitary", np.asarray(o.data), None)
    if isinstance(o, np.ndarray):
        if o.dtype == object: return ("seq", [make_sig(v, depth + 1) for v in o.tolist()])
        return ("array", o)
    if isinstance(o, dict):
        if o and all(isinstance(k, str) and set(k.replace(" ", "")) <= {"0", "1"} for k in o) \
           and all(isinstance(v, (int, float, np.integer, np.floating)) for v in o.values()):
            tot = float(sum(o.values()))
            if tot > 0:
                return ("dist", {k.replace(" ", ""): v / tot for k, v in o.items()})
        return ("map", {str(k): make_sig(v, depth + 1) for k, v in o.items()})
    if isinstance(o, (list, tuple)): return ("seq", [make_sig(v, depth + 1) for v in o])
    if OPEN_OBJECTS:
        s = _open_object(o)
        if s is not None: return s
    return ("opaque", type(o).__name__)

def is_strong(sig):
    """True if the fingerprint carries real information (not just 'some object of type X')."""
    k = sig[0]
    if k in ("opaque", "none", "error"): return False
    if k == "seq": return len(sig[1]) > 0 and all(is_strong(s) for s in sig[1])
    if k == "map": return len(sig[1]) > 0 and all(is_strong(s) for s in sig[1].values())
    return True

def _cos_equal(U, V, tol=1e-6):          # equality up to global phase
    U, V = np.asarray(U), np.asarray(V)
    if U.shape != V.shape: return False
    nu, nv = np.linalg.norm(U), np.linalg.norm(V)
    if nu == 0 or nv == 0: return bool(nu == nv)
    return abs(np.vdot(U, V)) / (nu * nv) > 1 - tol

def _sig_equal(a, b):
    if a[0] != b[0]: return False
    k = a[0]
    if k == "unitary":
        if a[2] != b[2]: return False                       # different number of measurements
        if CIRCUIT_EQ == "state" and a[2] is not None:      # circuit compared by its action on |0...0> (first column of U)
            return _cos_equal(a[1][:, 0], b[1][:, 0])
        return _cos_equal(a[1], b[1])
    if k == "state":   return _cos_equal(a[1], b[1])
    if k == "dist":
        p, q = a[1], b[1]
        return 0.5 * sum(abs(p.get(x, 0) - q.get(x, 0)) for x in set(p) | set(q)) <= DIST_TOL
    if k == "num":     return bool(np.isclose(a[1], b[1], rtol=1e-3, atol=1e-6))
    if k == "array":
        x, y = np.asarray(a[1]), np.asarray(b[1])
        if x.shape != y.shape: return False
        try: return bool(np.allclose(x, y, rtol=1e-3, atol=1e-6))
        except TypeError: return bool(np.array_equal(x, y))
    if k == "str":     return a[1] == b[1]
    if k == "none":    return True
    if k in ("struct", "opaque"): return a[1] == b[1]
    if k == "seq":     return len(a[1]) == len(b[1]) and all(_sig_equal(x, y) for x, y in zip(a[1], b[1]))
    if k == "map":     return a[1].keys() == b[1].keys() and all(_sig_equal(a[1][x], b[1][x]) for x in a[1])
    if k in ("dag", "scalarop", "instr", "plain"): return _plain_equal(a[1], b[1])
    return False       # "error" never equals anything

def sig_equal(a, b):
    try: return bool(_sig_equal(a, b))
    except Exception: return False

# ---------- running one program in a child process ----------
_ST = {"on": False, "rng": None, "aer": None, "n": 0}
def _seed_all(s=0):
    """Make the fingerprint run reproducible. Python/NumPy seeds are reset; numpy's default_rng() with no seed
    gets a per-program counter seed; Aer gets seed_simulator=0 ONLY when neither the call nor the backend options set one
    (programs that set their own seed must keep it)."""
    random.seed(s); np.random.seed(s)
    _ST["on"] = True; _ST["n"] = 0
    if _ST["rng"] is None:
        _ST["rng"] = np.random.default_rng
        def _det(seed=None):
            if seed is None and _ST["on"]:
                seed = _ST["n"] % (2**32); _ST["n"] += 1
            return _ST["rng"](seed)
        np.random.default_rng = _det
    try:
        from qiskit_aer.backends.aerbackend import AerBackend
        if _ST["aer"] is None:
            _ST["aer"] = AerBackend.run
            def _run(self, *a, **kw):
                if (_ST["on"] and kw.get("seed_simulator") is None
                        and getattr(self.options, "seed_simulator", None) is None):
                    kw["seed_simulator"] = 0
                return _ST["aer"](self, *a, **kw)
            AerBackend.run = _run
    except Exception:
        pass

def _unseed():
    """Switch the extra seeding off before the benchmark's own test runs (the test keeps its native behaviour)."""
    _ST["on"] = False

def _save_atomic(obj, path):
    """Write to a temporary file, then rename: a crash during a save can no longer truncate the cache."""
    import pickle, os
    with open(path + ".tmp", "wb") as f: pickle.dump(obj, f)
    os.replace(path + ".tmp", path)

def _sigs(fn, inputs):
    out = []
    for a, k in inputs:
        _seed_all(0)
        try:
            try: a2, k2 = copy.deepcopy(a), copy.deepcopy(k)
            except Exception: a2, k2 = a, k
            out.append(make_sig(fn(*a2, **k2)))
        except BaseException as e:
            return ("error", type(e).__name__)
    return ("seq", out)

def _materialize_inputs(srcs, entry_point, ns):
    """Turn call strings such as "f(QuantumCircuit(2), shots=10)" into (args, kwargs) pairs. They are evaluated inside the
    worker, in the candidate's own namespace (so the prompt's imports are available), with the entry-point name rebound
    to a function that just captures its arguments."""
    cap = lambda *a, **k: (a, k)
    out = []
    for s in srcs:
        _seed_all(0)
        out.append(eval(s, {**ns, entry_point: cap}))
    return out

def _run_job(job):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as _plt
    sink = io.StringIO()
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        ns = {"__name__": "__qhe__"}
        try:
            exec(compile(job["program"], "<program>", "exec"), ns)
            fn = ns[job["entry_point"]]
        except BaseException as e:
            return {"ok": False, "stage": "program", "error": f"{type(e).__name__}: {str(e)[:150]}"}
        try:
            exec(compile(job["test"], "<test>", "exec"), ns)
            check = ns["check"]
        except BaseException as e:
            return {"ok": False, "stage": "test_load", "error": f"{type(e).__name__}: {str(e)[:150]}"}
        res = {"ok": True, "passed": False}
        try:
            if job["mode"] == "record":       # canonical solution: record the inputs the test uses
                rec = []
                @functools.wraps(fn)
                def spy(*a, **k):
                    if len(rec) < 5:
                        try: rec.append((copy.deepcopy(a), copy.deepcopy(k)))
                        except Exception: rec.append((a, k))
                    return fn(*a, **k)
                try:
                    check(spy); res["passed"] = True
                except BaseException as e:
                    res["error"] = f"{type(e).__name__}: {str(e)[:150]}"
                res["inputs"] = rec if rec else [((), {})]
                res["sig"] = _sigs(fn, res["inputs"])
            else:                              # candidate: fingerprint first (fresh state), then run the real test
                try:
                    inputs = job["inputs"] if job.get("input_src") is None else _materialize_inputs(job["input_src"], job["entry_point"], ns)
                    res["sig"] = _sigs(fn, inputs)
                except BaseException as e:
                    res["sig"] = ("error", "InputBuild:" + type(e).__name__)
                if not job.get("skip_check"):
                    try:
                        _unseed(); check(fn); res["passed"] = True
                    except BaseException as e:
                        res["error"] = f"{type(e).__name__}: {str(e)[:150]}"
        finally:
            _plt.close("all")
        return res

def _child(job, conn):
    try:
        res = _run_job(job)
    except BaseException as e:
        res = {"ok": False, "stage": "harness", "error": f"{type(e).__name__}: {str(e)[:150]}"}
    try:
        conn.send(res)
    except BaseException:
        try: conn.send({"ok": False, "stage": "pickle", "error": "unpicklable result"})
        except BaseException: pass
    try: conn.close()
    finally: os._exit(0)

def run_jobs(jobs, n_workers=None, timeout=None, desc="running", progress=True):
    n_workers = n_workers or N_WORKERS; timeout = timeout or EXEC_TIMEOUT
    results = [None] * len(jobs)
    todo = list(range(len(jobs)))[::-1]
    running = {}
    bar = tqdm(total=len(jobs), desc=desc, disable=not progress)
    while todo or running:
        while todo and len(running) < n_workers:
            i = todo.pop()
            parent, child = ctx.Pipe(duplex=False)
            p = ctx.Process(target=_child, args=(jobs[i], child), daemon=True)
            p.start(); child.close()
            running[parent] = (i, p, time.time())
        for conn in mp_wait(list(running), timeout=0.2):
            i, p, _ = running.pop(conn)
            try: results[i] = conn.recv()
            except EOFError: results[i] = {"ok": False, "stage": "crash", "error": "worker crashed"}
            conn.close(); p.join(timeout=2)
            if p.is_alive(): p.kill()
            bar.update(1)
        now = time.time()
        for conn in list(running):
            i, p, t_start = running[conn]
            if now - t_start > timeout:
                p.kill(); p.join(); conn.close(); del running[conn]
                results[i] = {"ok": False, "stage": "timeout", "error": "timeout"}
                bar.update(1)
    bar.close()
    return results

## Step 3: which tasks work in *your* environment?
We run each task's own `check` against its **canonical solution**. Tasks that fail here (deprecated API, needs real IBM hardware, ...) are excluded and reported. This also records the call inputs used for fingerprinting.

In [ ]:
jobs = [dict(mode="record", program=build_program(t["prompt"], t["canonical_solution"]),
             test=t["test"], entry_point=t["entry_point"]) for t in tasks]
res = run_jobs(jobs, desc="validating canonical solutions")

valid_tasks, errs, excluded = {}, Counter(), {}
for t, r in zip(tasks, res):
    if r["ok"] and r["passed"] and r["sig"][0] != "error":
        t["inputs"], t["ref_sig"], t["strong"] = r["inputs"], r["sig"], is_strong(r["sig"])
        valid_tasks[t["task_id"]] = t
    else:
        msg = r.get("error") or (r["sig"][1] if r.get("sig") and r["sig"][0] == "error" else "")
        errs[(r.get("stage", "canonical_check"), msg[:90])] += 1
        excluded[t["task_id"]] = dict(stage=r.get("stage", "canonical_check"), error=msg[:150], difficulty=t["difficulty_scale"])

print(f"{len(valid_tasks)}/{len(tasks)} tasks usable; {sum(t['strong'] for t in valid_tasks.values())} of them have an informative fingerprint")
print(Counter(t["difficulty_scale"] for t in valid_tasks.values()))
print("\nWhy the others were excluded (top reasons):")
for (stage, msg), c in errs.most_common(8): print(f"  {c:3d} x [{stage}] {msg}")

### Optional: only if the cell above reports fewer than 138 usable tasks

In [ ]:
# OPTIONAL. Run only if the cell above reports fewer than 138 usable tasks.
# On a slower machine a few reference solutions can hit the 20-second limit. This re-checks ONLY those, with 60 s,
# and never touches the 13 tasks that are excluded in every run. Mention it in the paper if you needed it.
KNOWN_EXCLUDED = {f"qiskitHumanEval/{n}" for n in (43, 86, 94, 96, 97, 98, 104, 122, 129, 133, 134, 138, 146)}
slow = [t for t in tasks if excluded.get(t["task_id"], {}).get("stage") == "timeout" and t["task_id"] not in KNOWN_EXCLUDED]
print(len(valid_tasks), "usable;", len(slow), "extra time-outs to re-validate:", [t["task_id"] for t in slow])
if slow:
    jobs = [dict(mode="record", program=build_program(t["prompt"], t["canonical_solution"]), test=t["test"],
                 entry_point=t["entry_point"]) for t in slow]
    for t, r in zip(slow, run_jobs(jobs, desc="re-validating slow references", timeout=60)):
        if r["ok"] and r["passed"] and r["sig"][0] != "error":
            t["inputs"], t["ref_sig"], t["strong"] = r["inputs"], r["sig"], is_strong(r["sig"])
            valid_tasks[t["task_id"]] = t; excluded.pop(t["task_id"], None)
    print(len(valid_tasks), "usable after re-validation (expected 138)")


## Step 4: sample N completions per task
Plain left-to-right completion of the prompt (the prompt ends inside the function docstring), then cut at the end of the function body.
Sampling runs in a separate process (`gen.py`). Results are appended to `samples.jsonl` after every task, so after a Colab disconnect you can just re-run the last cell and it resumes.
Rough guess: on the order of an hour on a free T4 for all tasks with N=32 (depends on the GPU you get). Do a smoke test first: `MAX_TASKS = 10`, `N_SAMPLES = 8`.

In [ ]:
SAMPLES_PATH = f"{OUT_DIR}/samples.jsonl"
gen_tasks = [(i, t) for i, t in enumerate(tasks) if t["task_id"] in valid_tasks]
if MAX_TASKS: gen_tasks = gen_tasks[:MAX_TASKS]
json.dump([{"task_id": t["task_id"], "prompt": t["prompt"], "seed": SEED + i} for i, t in gen_tasks],
          open(f"{OUT_DIR}/gen_tasks.json", "w"))
json.dump(dict(model_id=MODEL_ID, n_samples=N_SAMPLES, temperature=TEMPERATURE, top_p=TOP_P,
               max_new_tokens=MAX_NEW_TOKENS, tasks_file=f"{OUT_DIR}/gen_tasks.json", out_file=SAMPLES_PATH),
          open(f"{OUT_DIR}/gen_config.json", "w"))
print(len(gen_tasks), "tasks will be sampled")

In [ ]:
%%writefile gen.py
# Generation runs as a separate process so this notebook's kernel never loads torch/CUDA
# (forking worker processes from a CUDA-initialised process is fragile).
import json, os, re, sys
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

cfg = json.load(open(sys.argv[1]))
jobs = json.load(open(cfg["tasks_file"]))
done = set()
if os.path.exists(cfg["out_file"]):
    for line in open(cfg["out_file"]):
        done.add(json.loads(line)["task_id"])
jobs = [j for j in jobs if j["task_id"] not in done]
print(f"{len(done)} tasks already sampled, {len(jobs)} to go", flush=True)
if not jobs: sys.exit(0)

tok = AutoTokenizer.from_pretrained(cfg["model_id"])
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
try:
    model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], dtype=torch.float16, device_map="auto")
except Exception:
    model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], torch_dtype=torch.float16, device_map="auto")
model.eval()

STOPS = cfg.get("stops") or ["\ndef ", "\nclass ", "\nif __name__", "\nprint(", "\n#", "\nimport ", "\nfrom ", "\n@", "\n```"]
GREEDY, DO_TRIM = bool(cfg.get("greedy", False)), cfg.get("trim", True)   # extra runs: greedy decoding / raw (untrimmed) text
def trim(text):
    m = re.search(r"\n(?=\S)", text)      # first line starting at column 0 = end of the function body
    return text[:m.start()] if m else text

with open(cfg["out_file"], "a") as f:
    for j in tqdm(jobs, desc="sampling"):
        torch.manual_seed(j["seed"])
        enc = tok(j["prompt"], return_tensors="pt").to(model.device)
        with torch.no_grad():
            if GREEDY:
                out = model.generate(**enc, do_sample=False, max_new_tokens=cfg["max_new_tokens"], num_return_sequences=1,
                                     stop_strings=STOPS, tokenizer=tok, pad_token_id=tok.pad_token_id)
            else:
                out = model.generate(**enc, do_sample=True, temperature=cfg["temperature"], top_p=cfg["top_p"],
                                     max_new_tokens=cfg["max_new_tokens"], num_return_sequences=cfg["n_samples"],
                                     stop_strings=STOPS, tokenizer=tok, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        comps = [trim(x) if DO_TRIM else x for x in texts]
        ntok = [len(tok(c, add_special_tokens=False).input_ids) for c in comps]
        f.write(json.dumps({"task_id": j["task_id"], "completions": comps, "n_tokens": ntok}) + "\n"); f.flush()
print("done", flush=True)

In [ ]:
!python gen.py {OUT_DIR}/gen_config.json

## Step 5: execute every candidate
For each candidate we record: did the real test pass (`passed`, used **only for scoring**), did it run and return something (`valid`), and its fingerprint. Then we build a task-level matrix `E[i, j]` = "candidate i and j give the same answer". Results are cached per task, so this cell is resumable too.

## Step 5a (v2.1): refresh cached results after changing the fingerprint
v2.1 reads the content of a few object types (circuit DAGs, scalar operators, instructions, noise models) instead of comparing them by type name only. Results cached by v2 hold only the type name for such outputs, so this cell drops those tasks from the cache and Step 5 then runs only them again (about 28 tasks, roughly 10-20 minutes). **Run Step 3 first** (it recomputes the reference fingerprints). To return to the v2 behaviour set `OPEN_OBJECTS = False` in the harness cell and use a fresh `OUT_DIR`.

In [ ]:
# Cached results from an older fingerprint contain only the TYPE NAME for objects such as circuit DAGs, scalar operators
# and noise models. Drop those tasks from the cache so that Step 5 executes them again with the content-based fingerprint.
def has_opaque(sig):
    if sig is None: return False
    k = sig[0]
    if k == "opaque": return True
    if k == "seq": return any(has_opaque(s) for s in sig[1])
    if k == "map": return any(has_opaque(s) for s in sig[1].values())
    return False

_p = f"{OUT_DIR}/exec_results.pkl"; _tag = "open-objects-v1" if OPEN_OBJECTS else "type-only"
if os.path.exists(_p):
    _c = pickle.load(open(_p, "rb")); _old = _c.get("__fingerprint__", "type-only")
    if _old == _tag:
        print(f"cache already uses the '{_tag}' fingerprint: {len([k for k in _c if k != '__fingerprint__'])} tasks kept, nothing to refresh")
    elif _tag == "open-objects-v1":
        stale = [t for t, rs in _c.items() if t != "__fingerprint__" and any(r.get("ok") and has_opaque(r["sig"]) for r in rs)]
        for t in stale: del _c[t]
        _c["__fingerprint__"] = _tag; _save_atomic(_c, _p)
        print(f"{len(stale)} tasks dropped from the cache (some outputs were compared by type only); Step 5 will run them again.")
    else:
        print("The cache was built with content-based fingerprints. To go back to type-only comparison use a fresh OUT_DIR.")
else:
    pickle.dump({"__fingerprint__": _tag}, open(_p, "wb"))      # tag the (empty) cache so later runs know which fingerprint built it
    print("no cache yet: Step 5 will run everything with the", _tag, "fingerprint")

In [ ]:
samples = {}
for line in open(SAMPLES_PATH):
    d = json.loads(line); samples[d["task_id"]] = d
run_ids = [tid for tid in samples if tid in valid_tasks]

EXEC_PATH = f"{OUT_DIR}/exec_results.pkl"
cache = pickle.load(open(EXEC_PATH, "rb")) if os.path.exists(EXEC_PATH) else {}

for k, tid in enumerate(tqdm(run_ids, desc="executing tasks")):
    if tid in cache: continue
    t = valid_tasks[tid]
    jobs = [dict(mode="candidate", program=build_program(t["prompt"], c), test=t["test"],
                 entry_point=t["entry_point"], inputs=t["inputs"])
            for c in samples[tid]["completions"][:N_SAMPLES]]
    cache[tid] = run_jobs(jobs, progress=False)
    if k % 10 == 0: _save_atomic(cache, EXEC_PATH)
_save_atomic(cache, EXEC_PATH)

def build_stats(tid):
    results = cache[tid]; N = len(results)
    passed = np.array([bool(r.get("ok") and r.get("passed")) for r in results])
    sigs = [r["sig"] if r.get("ok") else None for r in results]
    valid = np.array([s is not None and s[0] == "seq" and all(x[0] != "none" for x in s[1]) for s in sigs])
    E = np.zeros((N, N), bool)
    for i in range(N):
        if not valid[i]: continue
        E[i, i] = True
        for j in range(i + 1, N):
            if valid[j] and sig_equal(sigs[i], sigs[j]): E[i, j] = E[j, i] = True
    ref = valid_tasks[tid]["ref_sig"]
    matches_ref = np.array([bool(valid[i] and sig_equal(sigs[i], ref)) for i in range(N)])
    return dict(passed=passed, valid=valid, E=E, matches_ref=matches_ref,
                tokens=np.array(samples[tid]["n_tokens"][:N]),
                difficulty=valid_tasks[tid]["difficulty_scale"], strong=valid_tasks[tid]["strong"])

stats = {tid: build_stats(tid) for tid in run_ids}
print(len(stats), "tasks ready. Mean pass@1 =", round(float(np.mean([s["passed"].mean() for s in stats.values()])), 3))

## Step 6: selection strategies vs. number of samples N
For each N we average over 100 random subsets of the 32 samples per task.
* `random` - pick any sample (this is plain pass@1; computed exactly).
* `valid_filter` - pick a random sample among those that ran and returned something.
* `semantic` - cluster the valid samples by fingerprint, take the biggest cluster (ties split evenly), pick a random member.
* `oracle` - pass@N: at least one sample passes the real tests (what a perfect verifier would get; computed exactly with the unbiased pass@k formula).

In [ ]:
STRATS = ["random", "valid_filter", "semantic", "oracle"]
# A "view" = (which candidates count as valid, which equivalence matrix is used for clustering).
# "test" is the main method; "syntactic" (Step 11) and "llm" (Step 14) are baselines/ablations added later.
VIEWS = {"test": ("valid", "E"), "syntactic": ("valid", "E_syn"), "llm": ("valid_llm", "E_llm")}

def cluster_idx(idx, E):
    clusters = []
    for i in idx:
        for c in clusters:
            if E[c[0], i]: c.append(i); break
        else: clusters.append([i])
    return clusters

def sem_value(clusters, st, fallback):
    """Expected accuracy of the biggest cluster (ties split evenly, random member picked)."""
    if not clusters: return fallback
    m = max(len(c) for c in clusters)
    return float(np.mean([st["passed"][c].mean() for c in clusters if len(c) == m]))

def eval_subset(st, idx, valid_key="valid", E_key="E"):
    """Expected accuracy of the label-free strategies on ONE subset of the samples."""
    idx = np.asarray(idx)
    rand = float(st["passed"][idx].mean())
    vidx = [i for i in idx if st[valid_key][i]]
    filt = float(st["passed"][vidx].mean()) if vidx else rand
    sem = sem_value(cluster_idx(vidx, st[E_key]), st, rand)
    return [rand, filt, sem, float(st["passed"][idx].any())]

def pass_at_k_exact(N, c, k):
    """P(at least one of k samples drawn without replacement from N is correct) when c of the N are correct."""
    k = min(k, N)
    return 1.0 if N - c < k else 1.0 - math.comb(N - c, k) / math.comb(N, k)

NS = [n for n in (1, 2, 4, 8, 12, 16, 24, 32, 64) if n <= N_SAMPLES]

def fixed_pertask(task_ids, ns=None, R=100, seed=0, st_all=None, views=("test",)):
    """Per-task expected accuracy of every strategy at every N.
    `random` and `oracle` are exact (closed form), and so is everything at N=1 and N=N_SAMPLES. For 1 < N < N_SAMPLES `valid_filter` and `semantic` are averaged over R random subsets;
    the SAME subsets are used for every view, so differences between views are paired."""
    ns = list(ns or NS); st_all = st_all or stats
    rng = np.random.default_rng(seed); T = len(task_ids)
    sfx = lambda v: "" if v == "test" else f"_{v}"
    names = ["random", "oracle"] + [f"valid_filter{sfx(v)}" for v in views] + [f"semantic{sfx(v)}" for v in views]
    vals = {k: np.zeros((T, len(ns))) for k in names}; toks = np.zeros((T, len(ns)))
    for a, tid in enumerate(task_ids):
        st = st_all[tid]; N = len(st["passed"]); c = int(st["passed"].sum())
        for b, n in enumerate(ns):
            vals["random"][a, b] = st["passed"].mean()
            vals["oracle"][a, b] = pass_at_k_exact(N, c, n)
            if n >= N:   draws = [np.arange(N)]                          # the whole set: exact
            elif n == 1: draws = [np.array([i]) for i in range(N)]       # every single sample: exact (all strategies = pass@1)
            else:        draws = [rng.permutation(N)[:n] for _ in range(R)]
            for v in views:
                vk, ek = VIEWS[v]
                filt, sem = np.array([eval_subset(st, idx, vk, ek)[1:3] for idx in draws]).mean(axis=0)
                vals[f"valid_filter{sfx(v)}"][a, b], vals[f"semantic{sfx(v)}"][a, b] = filt, sem
            toks[a, b] = min(n, N) * st["tokens"].mean()
    return {"ns": ns, "ids": list(task_ids), "vals": vals, "tokens": toks}

def fixed_curve(task_ids, R=100, seed=0, return_pt=False, **kw):
    pt = fixed_pertask(task_ids, R=R, seed=seed, **kw)
    df = pd.DataFrame({"n": pt["ns"], **{s: pt["vals"][s].mean(axis=0) for s in STRATS}, "tokens": pt["tokens"].mean(axis=0)})
    return (df, pt) if return_pt else df

all_ids    = list(stats)
strong_ids = [t for t in all_ids if stats[t]["strong"]]
curve_all,    pt_all    = fixed_curve(all_ids,    return_pt=True)
curve_strong, pt_strong = fixed_curve(strong_ids, return_pt=True)
print(f"ALL usable tasks (n={len(all_ids)})"); display(curve_all.round(3))
print(f"Tasks with an informative fingerprint (n={len(strong_ids)})"); display(curve_strong.round(3))

## Step 7: adaptive stopping
Sample one program at a time in random order. Stop as soon as the leading cluster has at least `tau` members **and** is strictly bigger than the runner-up (or we run out of samples). Easy tasks stop after a few samples; hard or ambiguous tasks keep sampling.

In [ ]:
def beta_conf(v1, v2):
    """Adaptive-Consistency's Beta stopping score: P(Beta(v1+1, v2+1) > 1/2), i.e. how sure we are that the leading
    answer (v1 votes) really is more likely than the runner-up (v2 votes). Exact, via the binomial identity."""
    a, b = v1 + 1, v2 + 1; m = a + b - 1
    return sum(math.comb(m, k) for k in range(a)) / 2 ** m

def should_stop(sizes, n, rule, param, min_n):
    if n < min_n or not sizes: return False
    top, second = sizes[0], (sizes[1] if len(sizes) > 1 else 0)
    if rule == "tau":  return top >= param and top > second          # our rule
    if rule == "beta": return beta_conf(top, second) >= param        # Adaptive-Consistency (Aggarwal et al., 2023)
    raise ValueError(rule)

def adaptive_pertask(task_ids, rule="tau", param=3, R=100, seed=1, min_n=2, st_all=None, view="test"):
    """Sample one program at a time in random order; stop with `rule`; answer = biggest cluster so far.
    Returns per-task arrays (expected accuracy, number of samples used, tokens used). Same random orders for every
    rule/param (same seed), so rules can be compared pairwise."""
    st_all = st_all or stats; vk, ek = VIEWS[view]
    rng = np.random.default_rng(seed); T = len(task_ids)
    acc, ns_, tok = np.zeros(T), np.zeros(T), np.zeros(T)
    for a, tid in enumerate(task_ids):
        st = st_all[tid]; N = len(st["passed"]); E = st[ek]; a_t, n_t, k_t = [], [], []
        for _ in range(R):
            order = rng.permutation(N); clusters = []
            for n, i in enumerate(order, 1):
                if st[vk][i]:
                    for c in clusters:
                        if E[c[0], i]: c.append(i); break
                    else: clusters.append([i])
                sizes = sorted((len(c) for c in clusters), reverse=True)
                if should_stop(sizes, n, rule, param, min_n): break
            idx = order[:n]
            a_t.append(sem_value(clusters, st, float(st["passed"][idx].mean())))
            n_t.append(n); k_t.append(st["tokens"][idx].sum())
        acc[a], ns_[a], tok[a] = np.mean(a_t), np.mean(n_t), np.mean(k_t)
    return {"acc": acc, "n": ns_, "tokens": tok}

def adaptive_curve(task_ids, taus=(2, 3, 4, 5, 6, 8), R=100, seed=1, min_n=2, rule="tau"):
    rows = []
    for p in taus:
        r = adaptive_pertask(task_ids, rule, p, R, seed, min_n)
        rows.append({("tau" if rule == "tau" else "conf"): p, "semantic_adaptive": float(r["acc"].mean()),
                     "avg_samples": float(r["n"].mean()), "tokens": float(r["tokens"].mean())})
    return pd.DataFrame(rows)

adapt_all, adapt_strong = adaptive_curve(all_ids), adaptive_curve(strong_ids)
print("Adaptive stopping, ALL usable tasks"); display(adapt_all.round(3))
print("Adaptive stopping, informative-fingerprint tasks"); display(adapt_strong.round(3))

## Step 8: figures for the paper

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
style = {"random": ("tab:gray", "o"), "valid_filter": ("tab:blue", "s"), "semantic": ("tab:red", "D"), "oracle": ("black", "^")}
for col, (name, cv, ad) in enumerate([("All usable tasks", curve_all, adapt_all),
                                      ("Informative-fingerprint tasks", curve_strong, adapt_strong)]):
    ax = axes[0, col]
    for s in STRATS:
        ax.plot(cv["n"], 100 * cv[s], marker=style[s][1], color=style[s][0], label=s,
                ls="--" if s == "oracle" else "-")
    ax.set_xscale("log", base=2); ax.set_xticks(cv["n"]); ax.set_xticklabels(cv["n"])
    ax.set_xlabel("samples per problem (N)"); ax.set_ylabel("accuracy (%)"); ax.set_title(name); ax.grid(alpha=.3)
    if col == 0: ax.legend()
    ax = axes[1, col]
    ax.plot(cv["tokens"], 100 * cv["semantic"], "D-", color="tab:red", label="semantic, fixed N")
    ax.plot(cv["tokens"], 100 * cv["random"], "o-", color="tab:gray", label="random (pass@1)")
    ax.plot(ad["tokens"], 100 * ad["semantic_adaptive"], "*", color="tab:green", ms=14, ls="-", label="semantic, adaptive stop")
    for k, (_, r) in enumerate(ad.iterrows()): ax.annotate(f"tau={int(r.tau)}", (r.tokens, 100 * r.semantic_adaptive), fontsize=7, xytext=(4, -11 if k % 2 == 0 else 7), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("avg generated tokens per problem"); ax.set_ylabel("accuracy (%)"); ax.grid(alpha=.3)
    if col == 0: ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figures.png", dpi=200); plt.show()
curve_all.to_csv(f"{OUT_DIR}/curve_all.csv", index=False); curve_strong.to_csv(f"{OUT_DIR}/curve_strong.csv", index=False)
adapt_all.to_csv(f"{OUT_DIR}/adaptive_all.csv", index=False); adapt_strong.to_csv(f"{OUT_DIR}/adaptive_strong.csv", index=False)

## Step 9: diagnostics (this is where the discussion section comes from)

In [ ]:
# 1) Accuracy by difficulty at the largest N
rows = []
for diff in sorted({stats[t]["difficulty"] for t in all_ids}):
    ids = [t for t in all_ids if stats[t]["difficulty"] == diff]
    c = fixed_curve(ids, R=50).iloc[[0, -1]]
    rows.append({"difficulty": diff, "tasks": len(ids),
                 "pass@1": c.iloc[0]["random"], f"semantic@{c.iloc[1]['n']:.0f}": c.iloc[1]["semantic"],
                 f"oracle@{c.iloc[1]['n']:.0f}": c.iloc[1]["oracle"]})
print("By difficulty"); display(pd.DataFrame(rows).round(3))

# 2) How good is "agrees with the reference's output" as a proxy for "passes the property tests"?
tp = fp = fn = tn = 0
for tid in all_ids:
    st = stats[tid]; v = st["valid"]
    m, p = st["matches_ref"][v], st["passed"][v]
    tp += int((m & p).sum()); fp += int((m & ~p).sum()); fn += int((~m & p).sum()); tn += int((~m & ~p).sum())
print("Among candidates that ran and returned a value:")
print(f"  match canonical output & pass tests : {tp}\n  match canonical output & FAIL tests : {fp}\n"
      f"  differ from canonical & pass tests  : {fn}  <- correct but 'different' (tests are property checks)\n"
      f"  differ from canonical & fail tests  : {tn}")

# 3) Coverage
print(f"\nCoverage: {len(tasks)} tasks in dataset -> {len(valid_tasks)} usable -> {len(all_ids)} sampled -> "
      f"{len(strong_ids)} with an informative fingerprint")
print("Tasks where at least one sample passed:", sum(stats[t]['passed'].any() for t in all_ids))

## Step 10: statistics for the paper (CPU only)
Steps 10-12 re-use `samples.jsonl` and `exec_results.pkl`; **nothing is regenerated**. Confidence intervals come from a **bootstrap over tasks** (the unit of replication), with the same resamples for every statistic, so differences between methods are paired. `random` (= pass@1) and `oracle` (= pass@N) are exact; `valid_filter` and `semantic` are averaged over random subsets of the samples.

**10a** confidence intervals, paired differences, and the share of the random-to-oracle gap that is closed.

In [ ]:
import ast
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 250)

# ---- bootstrap over TASKS (the unit of replication); the same resamples are reused for every statistic => paired ----
def boot_idx(T, B=None, seed=12345):
    return np.random.default_rng(seed).integers(0, T, size=(B or BOOT_B, T))

def boot_mean(x, idx):                       # x: (T,) per-task values -> (B,) bootstrap means
    return np.asarray(x)[idx].mean(axis=1)

def fmt_ci(x, idx, scale=100, nd=1):
    bs = boot_mean(x, idx); lo, hi = np.percentile(bs, [2.5, 97.5])
    return f"{scale * np.mean(x):.{nd}f} [{scale * lo:.{nd}f}, {scale * hi:.{nd}f}]"

def paired_diff(a, b, idx):
    """mean difference, 95% bootstrap CI, two-sided bootstrap p-value (a - b, per task)."""
    d = np.asarray(a) - np.asarray(b); bs = boot_mean(d, idx); lo, hi = np.percentile(bs, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi), float(min(1.0, 2 * min((bs <= 0).mean(), (bs >= 0).mean())))

def ci_table(pt, keys=STRATS):
    idx = boot_idx(len(pt["ids"]))
    return pd.DataFrame([{"n": n, **{k: fmt_ci(pt["vals"][k][:, j], idx) for k in keys}} for j, n in enumerate(pt["ns"])])

def diff_table(pt, pairs):
    idx = boot_idx(len(pt["ids"])); rows = []
    for j, n in enumerate(pt["ns"]):
        row = {"n": n}
        for a, b in pairs:
            m, lo, hi, p = paired_diff(pt["vals"][a][:, j], pt["vals"][b][:, j], idx)
            row[f"{a} - {b}"] = f"{100*m:+.1f} [{100*lo:+.1f}, {100*hi:+.1f}] " + ("p<0.001" if p < 0.001 else f"p={p:.3f}")
        rows.append(row)
    return pd.DataFrame(rows)

# ---- shared helpers for building equivalence matrices from cached execution results ----
def valid_mask(sigs):
    return np.array([s is not None and s[0] == "seq" and all(x[0] != "none" for x in s[1]) for s in sigs])

def equiv_matrix(sigs, valid):
    N = len(sigs); E = np.zeros((N, N), bool)
    for i in range(N):
        if not valid[i]: continue
        E[i, i] = True
        for j in range(i + 1, N):
            if valid[j] and sig_equal(sigs[i], sigs[j]): E[i, j] = E[j, i] = True
    return E

# ---- tables ----
print("Accuracy (%) with 95% bootstrap CIs over tasks")
for name, pt in [("ALL usable tasks", pt_all), ("Informative-fingerprint tasks", pt_strong)]:
    print(f"\n{name} (n={len(pt['ids'])})"); display(ci_table(pt))

PAIRS = [("semantic", "valid_filter"), ("semantic", "random"), ("valid_filter", "random"), ("oracle", "semantic")]
print("\nPaired differences in accuracy (percentage points), ALL usable tasks")
display(diff_table(pt_all, PAIRS))

def gap_stats(pt, j=-1):
    """Share of the random->oracle gap closed, and how much of semantic's gain comes from merely dropping non-running programs."""
    idx = boot_idx(len(pt["ids"])); v = {k: pt["vals"][k][:, j] for k in STRATS}
    def ratio(num, den):
        pt_est = num.mean() / den.mean()
        with np.errstate(divide="ignore", invalid="ignore"):
            bs = boot_mean(num, idx) / boot_mean(den, idx)
        lo, hi = np.nanpercentile(bs, [2.5, 97.5]); return f"{100*pt_est:.0f}% [{100*lo:.0f}%, {100*hi:.0f}%]"
    gap = v["oracle"] - v["random"]
    return {"gap closed by semantic": ratio(v["semantic"] - v["random"], gap),
            "gap closed by valid_filter": ratio(v["valid_filter"] - v["random"], gap),
            "share of semantic's gain that is just filtering": ratio(v["valid_filter"] - v["random"], v["semantic"] - v["random"])}

for name, pt in [("ALL usable tasks", pt_all), ("Informative-fingerprint tasks", pt_strong)]:
    print(f"\n{name}, at N={pt['ns'][-1]}:")
    for k, s in gap_stats(pt).items(): print(f"  {k:48s}: {s}")

**10b** what happens to the candidates (this decomposes where `valid_filter`'s gain comes from), and which tasks produce candidates that match the canonical output but fail the tests.

In [ ]:
# 1) What happens to the candidates? (explains where `valid_filter`'s gain comes from)
def candidate_category(r):
    if not r.get("ok"):
        stage, err = r.get("stage"), (r.get("error") or "")
        if stage == "timeout": return f"timeout ({EXEC_TIMEOUT} s)"
        if stage == "program": return "program failed to load: " + (err.split(":")[0] or "?")
        return f"harness: {stage}"
    s = r["sig"]
    if s[0] == "error": return "call raised " + str(s[1])
    if s[0] == "seq" and any(x[0] == "none" for x in s[1]): return "returned None"
    return "ran and returned a value"

cat_n, cat_pass = Counter(), Counter()
for tid in all_ids:
    for r in cache[tid][:N_SAMPLES]:
        c = candidate_category(r); cat_n[c] += 1; cat_pass[c] += int(bool(r.get("ok") and r.get("passed")))
tot = sum(cat_n.values())
df_cat = pd.DataFrame([{"outcome": c, "candidates": n, "share %": round(100 * n / tot, 1), "passed the tests": cat_pass[c]}
                       for c, n in cat_n.most_common()])
print(f"What happened to the {tot} candidates?"); display(df_cat.head(15))
df_cat.to_csv(f"{OUT_DIR}/candidate_outcomes.csv", index=False)

# 2) Candidates that match the canonical output but fail the real test: concentrated in a few tasks?
rows = []
for tid in all_ids:
    st = stats[tid]; m, p = st["matches_ref"], st["passed"]
    n_fail, n_pass = int((m & ~p).sum()), int((m & p).sum())
    if n_fail:
        errs_ = Counter((cache[tid][i].get("error") or "")[:70] for i in np.where(m & ~p)[0])
        rows.append({"task_id": tid, "difficulty": st["difficulty"], "match&pass": n_pass, "match&FAIL": n_fail,
                     "most common error": errs_.most_common(1)[0][0]})
if rows:
    df_mf = pd.DataFrame(rows).sort_values("match&FAIL", ascending=False).reset_index(drop=True)
    total_fail = int(df_mf["match&FAIL"].sum()); top5 = int(df_mf["match&FAIL"].head(5).sum())
    print(f"\n{total_fail} candidates match the canonical output but fail the tests, spread over {len(df_mf)} tasks; "
          f"the top 5 tasks account for {100 * top5 / total_fail:.0f}% of them.")
    display(df_mf.head(12)); df_mf.to_csv(f"{OUT_DIR}/match_but_fail.csv", index=False)
else:
    print("\nNo candidate matches the canonical output and fails the tests.")

**10c** accuracy by difficulty. Check the task counts: some buckets are tiny, so their intervals are wide.

In [ ]:
# Accuracy (%) at the largest N by difficulty, with bootstrap CIs. Report task COUNTS: some buckets are tiny.
diffs = np.array([stats[t]["difficulty"] for t in pt_all["ids"]]); rows = []
for d in sorted(set(diffs)):
    m = diffs == d; idx = boot_idx(int(m.sum()))
    gc = (pt_all["vals"]["semantic"][m, -1].mean() - pt_all["vals"]["random"][m, -1].mean()) / \
         max(pt_all["vals"]["oracle"][m, -1].mean() - pt_all["vals"]["random"][m, -1].mean(), 1e-12)
    rows.append({"difficulty": d, "tasks": int(m.sum()), **{s: fmt_ci(pt_all["vals"][s][m, -1], idx) for s in STRATS},
                 "gap closed": f"{100 * gc:.0f}%"})
df_diff = pd.DataFrame(rows)
print(f"Accuracy (%) at N={pt_all['ns'][-1]} by difficulty, 95% bootstrap CIs"); display(df_diff)
df_diff.to_csv(f"{OUT_DIR}/by_difficulty_ci.csv", index=False)

## Step 11: baselines and ablations (CPU only)
* **Syntactic majority vote**: cluster the valid candidates by normalised program text instead of by what they compute. If `semantic` does not beat this, execution adds nothing.
* **Equivalence ablation**: unitary vs. state equivalence, and the total-variation tolerance for counts.
* **No-argument tasks**: the function is called without arguments, so no test-derived input is used at all.
Needs Step 10a (helpers).

In [ ]:
class _CollectNames(ast.NodeVisitor):
    def __init__(self): self.order, self.params = [], set()
    def visit_Name(self, n):
        if isinstance(n.ctx, ast.Store) and n.id not in self.order: self.order.append(n.id)
    def visit_arg(self, n): self.params.add(n.arg)

class _RenameNames(ast.NodeTransformer):
    def __init__(self, m): self.m = m
    def visit_Name(self, n):
        if n.id in self.m: n.id = self.m[n.id]
        return n

def syntactic_key(prompt, completion, rename=True):
    """Canonical form of a program: parsed AST without comments/whitespace/docstrings, local variables renamed
    in order of first assignment (so `qc = ...` and `circuit = ...` count as the same program)."""
    try: tree = ast.parse(build_program(prompt, completion))
    except SyntaxError: return "RAW:" + re.sub(r"\s+", " ", completion.strip())
    for node in ast.walk(tree):
        if isinstance(node, (ast.Module, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.body \
           and isinstance(node.body[0], ast.Expr) and isinstance(getattr(node.body[0], "value", None), ast.Constant) \
           and isinstance(node.body[0].value.value, str):
            node.body = node.body[1:] or [ast.Pass()]
    if rename:
        c = _CollectNames(); c.visit(tree)
        tree = _RenameNames({n: f"_v{i}" for i, n in enumerate(x for x in c.order if x not in c.params)}).visit(tree)
    return ast.dump(tree, annotate_fields=False, include_attributes=False)

for tid in all_ids:
    st = stats[tid]; t = valid_tasks[tid]
    keys = [syntactic_key(t["prompt"], c) for c in samples[tid]["completions"][:len(st["passed"])]]
    lab = {}; labels = np.array([lab.setdefault(k, len(lab)) for k in keys])
    st["E_syn"] = (labels[:, None] == labels[None, :]) & st["valid"][:, None] & st["valid"][None, :]
    st["n_syn_clusters"] = len({labels[i] for i in np.where(st["valid"])[0]})

pt_syn = fixed_pertask(all_ids, views=("test", "syntactic"))
print("Semantic (execution-based) vs syntactic (same program text) majority vote, ALL usable tasks")
display(ci_table(pt_syn, ["random", "valid_filter", "semantic_syntactic", "semantic", "oracle"]))
print("Paired differences (percentage points)")
display(diff_table(pt_syn, [("semantic", "semantic_syntactic"), ("semantic_syntactic", "valid_filter")]))
sem_cl = np.mean([len(cluster_idx(np.where(stats[t]["valid"])[0], stats[t]["E"])) for t in all_ids])
syn_cl = np.mean([stats[t]["n_syn_clusters"] for t in all_ids])
print(f"Average number of clusters among the valid candidates of a task: semantic {sem_cl:.1f}, syntactic {syn_cl:.1f} (of {N_SAMPLES} samples)")

In [ ]:
# Table 3: code matching vs behaviour matching on the informative tasks
pt_syn_strong = fixed_pertask(strong_ids, views=("test", "syntactic"))
print(f"Informative-fingerprint tasks (n={len(strong_ids)})")
display(ci_table(pt_syn_strong, ["valid_filter", "semantic_syntactic", "semantic"]))
display(diff_table(pt_syn_strong, [("semantic", "semantic_syntactic"), ("semantic_syntactic", "valid_filter")]))


In [ ]:
# Table 9: agreement with the reference output, informative vs other tasks
S = set(strong_ids); rows = []
for name, ids in (("informative", [t for t in all_ids if t in S]), ("other", [t for t in all_ids if t not in S])):
    tp = fp = fn = tn = 0
    for tid in ids:
        st = stats[tid]; v = st["valid"]; m, p = st["matches_ref"][v], st["passed"][v]
        tp += int((m & p).sum()); fp += int((m & ~p).sum()); fn += int((~m & p).sum()); tn += int((~m & ~p).sum())
    rows.append({"tasks": name, "n_tasks": len(ids), "valid": tp+fp+fn+tn, "match_pass": tp, "match_fail": fp,
                 "differ_pass": fn, "differ_fail": tn,
                 "P(pass|match)": f"{100*tp/max(tp+fp,1):.1f}%", "P(pass|differ)": f"{100*fn/max(fn+tn,1):.1f}%"})
df_t9 = pd.DataFrame(rows); display(df_t9); df_t9.to_csv(f"{OUT_DIR}/table9_split.csv", index=False)


In [ ]:
# Ablation: how sensitive is `semantic` to the definition of "same answer"?  (uses only cached execution results)
def stats_under(circuit_eq, dist_tol):
    global CIRCUIT_EQ, DIST_TOL
    keep = (CIRCUIT_EQ, DIST_TOL); CIRCUIT_EQ, DIST_TOL = circuit_eq, dist_tol
    try:
        out = {}
        for tid in all_ids:
            sigs = [r["sig"] if r.get("ok") else None for r in cache[tid][:N_SAMPLES]]
            out[tid] = {**stats[tid], "E": equiv_matrix(sigs, valid_mask(sigs))}
        return out
    finally:
        CIRCUIT_EQ, DIST_TOL = keep

SETTINGS = {f"{CIRCUIT_EQ}, tol {DIST_TOL} (default)": (CIRCUIT_EQ, DIST_TOL), "state, tol 0.1": ("state", 0.1),
            "unitary, tol 0.05": ("unitary", 0.05), "unitary, tol 0.2": ("unitary", 0.2)}
n_eval = sorted({n for n in (8, N_SAMPLES) if n <= N_SAMPLES}); idx_b = boot_idx(len(all_ids)); base = None; rows = []
for name, (ce, dt) in SETTINGS.items():
    pt = fixed_pertask(all_ids, ns=n_eval, st_all=stats_under(ce, dt))
    base = pt["vals"]["semantic"] if base is None else base
    row = {"equivalence": name}
    for j, n in enumerate(n_eval):
        m, lo, hi, _ = paired_diff(pt["vals"]["semantic"][:, j], base[:, j], idx_b)
        row[f"semantic@{n}"] = fmt_ci(pt["vals"]["semantic"][:, j], idx_b)
        row[f"vs default @{n}"] = f"{100*m:+.1f} [{100*lo:+.1f}, {100*hi:+.1f}]"
    rows.append(row)
print("Ablation of the equivalence definition (accuracy %, 95% CI over tasks)"); display(pd.DataFrame(rows))

# Tasks whose fingerprint needs NO input at all (the function is called without arguments): no test-derived input is used.
def is_noarg(inputs): return all(len(a) == 0 and len(k) == 0 for a, k in inputs)
noarg_ids = [t for t in all_ids if is_noarg(valid_tasks[t]["inputs"])]
args_ids  = [t for t in all_ids if t not in set(noarg_ids)]
print(f"\nNo-argument tasks: {len(noarg_ids)} of {len(all_ids)} (the fingerprint uses no test-derived input here)")
for name, ids in [("NO-ARGUMENT tasks", noarg_ids), ("tasks WITH arguments", args_ids)]:
    if len(ids) < 3: print(f"{name}: only {len(ids)} tasks, skipped"); continue
    pt = fixed_pertask(ids, ns=[1, n_eval[-1]])
    print(f"\n{name} (n={len(ids)}), accuracy % [95% CI]"); display(ci_table(pt))

## Step 12: adaptive stopping, done properly (CPU only)
1. Our rule (`tau`) next to **Adaptive-Consistency's Beta rule** (Aggarwal et al., 2023): stop when P(leader's true probability > runner-up's) >= a confidence level.
2. Each adaptive setting is compared with **fixed N at the same average number of samples**, not just with the largest N.
3. The threshold is **chosen on training folds and evaluated on held-out tasks** (5-fold cross-validation over tasks), so the reported accuracy is not tuned on the tasks it is measured on.

In [ ]:
TAUS  = tuple(p for p in (2, 3, 4, 5, 6, 8) if p <= N_SAMPLES)
CONFS = (0.8, 0.9, 0.95, 0.99)
ad_tau  = {p: adaptive_pertask(all_ids, "tau",  p) for p in TAUS}
ad_beta = {c: adaptive_pertask(all_ids, "beta", c) for c in CONFS}

full = fixed_pertask(all_ids, ns=[N_SAMPLES]); sem_full, tok_full = full["vals"]["semantic"][:, 0], full["tokens"][:, 0]
_fixed_cache = {}
def fixed_at(n):                                   # fixed-N semantic accuracy per task (cached)
    if n not in _fixed_cache: _fixed_cache[n] = fixed_pertask(all_ids, ns=[n])["vals"]["semantic"][:, 0]
    return _fixed_cache[n]

idx_a = boot_idx(len(all_ids)); rows = []
for rule, d in (("tau", ad_tau), ("beta", ad_beta)):
    for p, r in d.items():
        n_match = max(1, int(round(r["n"].mean())))
        m1 = paired_diff(r["acc"], fixed_at(n_match), idx_a); m2 = paired_diff(r["acc"], sem_full, idx_a)
        rows.append({"rule": f"{rule}={p}", "avg samples": round(r["n"].mean(), 1),
                     "tokens saved vs N": f"{100 * (1 - r['tokens'].mean() / tok_full.mean()):.0f}%",
                     "adaptive acc": fmt_ci(r["acc"], idx_a), "matched N": n_match,
                     "fixed acc (matched N)": fmt_ci(fixed_at(n_match), idx_a),
                     "adaptive - fixed(matched N)": f"{100*m1[0]:+.1f} [{100*m1[1]:+.1f}, {100*m1[2]:+.1f}]",
                     f"adaptive - fixed N={N_SAMPLES}": f"{100*m2[0]:+.1f} [{100*m2[1]:+.1f}, {100*m2[2]:+.1f}]"})
df_match = pd.DataFrame(rows)
print(f"Adaptive stopping vs fixed N at the same average number of samples (ALL usable tasks; fixed N={N_SAMPLES}: "
      f"{100 * sem_full.mean():.1f}% with {tok_full.mean():.0f} tokens)")
display(df_match); df_match.to_csv(f"{OUT_DIR}/adaptive_matched.csv", index=False)

In [ ]:
def cv_select(d, K=5, reps=20, delta=0.01, seed=0):
    """Choose the stopping threshold on TRAINING tasks only: the cheapest setting (fewest tokens) whose training accuracy
    is within `delta` of fixed N. Evaluate that choice on the held-out fold. Repeated K-fold, averaged over repetitions."""
    keys = list(d); T = len(all_ids); rng = np.random.default_rng(seed)
    out = {k: np.zeros(T) for k in ("acc", "n", "tokens")}; chosen = Counter()
    for _ in range(reps):
        for fold in np.array_split(rng.permutation(T), K):
            train = np.setdiff1d(np.arange(T), fold)
            ok = [k for k in keys if d[k]["acc"][train].mean() >= sem_full[train].mean() - delta]
            k_star = min(ok, key=lambda k: d[k]["tokens"][train].mean()) if ok else max(keys)   # fallback: most conservative
            chosen[k_star] += 1
            for q in out: out[q][fold] += d[k_star][q][fold] / reps
    return out, chosen

rows = []
for rule, d in (("tau", ad_tau), ("beta", ad_beta)):
    r, chosen = cv_select(d)
    m = paired_diff(r["acc"], sem_full, idx_a)
    rows.append({"rule": rule, "threshold chosen (share of folds)": ", ".join(f"{k}: {100*v/sum(chosen.values()):.0f}%" for k, v in sorted(chosen.items())),
                 "held-out acc": fmt_ci(r["acc"], idx_a), "avg samples": round(r["n"].mean(), 1),
                 "tokens saved vs N": f"{100 * (1 - r['tokens'].mean() / tok_full.mean()):.0f}%",
                 f"vs fixed N={N_SAMPLES}": f"{100*m[0]:+.1f} [{100*m[1]:+.1f}, {100*m[2]:+.1f}]"})
df_cv = pd.DataFrame(rows)
print("Stopping threshold chosen by 5-fold cross-validation over tasks (cheapest setting within 1 point of fixed N on the training folds)")
display(df_cv); df_cv.to_csv(f"{OUT_DIR}/adaptive_cv.csv", index=False)

## Step 13 (GPU, optional): greedy-decoding baseline
Is sampling 32 programs and voting better than simply decoding greedily once? This generates one greedy completion per task (`do_sample=False`) with the same model and compares it, task by task, with the sampling-based strategies. Set `RUN_GREEDY = False` to skip it.

In [ ]:
def prepare_generation(name, jobs, **overrides):
    """Write the task file + config for one extra gen.py run (resumable) and return the config path."""
    cfg = dict(model_id=MODEL_ID, n_samples=N_SAMPLES, temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS)
    cfg.update(overrides)
    cfg["tasks_file"], cfg["out_file"] = f"{OUT_DIR}/{name}_tasks.json", f"{OUT_DIR}/{name}.jsonl"
    json.dump(jobs, open(cfg["tasks_file"], "w")); path = f"{OUT_DIR}/{name}_config.json"; json.dump(cfg, open(path, "w"))
    return path

if RUN_GREEDY:
    GREEDY_CFG = prepare_generation("greedy", [{"task_id": t, "prompt": valid_tasks[t]["prompt"], "seed": SEED} for t in run_ids],
                                    greedy=True, n_samples=1)
    print("config written:", GREEDY_CFG)

In [ ]:
if RUN_GREEDY:
    !python gen.py {GREEDY_CFG}

In [ ]:
if RUN_GREEDY:
    greedy = {}
    for line in open(f"{OUT_DIR}/greedy.jsonl"): d = json.loads(line); greedy[d["task_id"]] = d
    g_ids = [t for t in all_ids if t in greedy]
    G_PATH = f"{OUT_DIR}/greedy_exec.pkl"
    cache_g = pickle.load(open(G_PATH, "rb")) if os.path.exists(G_PATH) else {}
    todo = [t for t in g_ids if t not in cache_g]
    jobs = [dict(mode="candidate", program=build_program(valid_tasks[t]["prompt"], greedy[t]["completions"][0]), test=valid_tasks[t]["test"],
                 entry_point=valid_tasks[t]["entry_point"], inputs=valid_tasks[t]["inputs"]) for t in todo]
    for t, r in zip(todo, run_jobs(jobs, desc="executing greedy programs")): cache_g[t] = r
    _save_atomic(cache_g, G_PATH)

    pos = {t: i for i, t in enumerate(pt_all["ids"])}; sel = [pos[t] for t in g_ids]
    ref = {"greedy decoding (1 sample)": np.array([float(bool(cache_g[t].get("ok") and cache_g[t].get("passed"))) for t in g_ids]),
           "sampled pass@1 (T=%.1f)" % TEMPERATURE: pt_all["vals"]["random"][sel, 0],
           f"valid_filter@{N_SAMPLES}": pt_all["vals"]["valid_filter"][sel, -1],
           f"semantic@{N_SAMPLES}": pt_all["vals"]["semantic"][sel, -1],
           f"oracle@{N_SAMPLES}": pt_all["vals"]["oracle"][sel, -1]}
    idx = boot_idx(len(g_ids)); g_tok = np.mean([greedy[t]["n_tokens"][0] for t in g_ids])
    print(f"Greedy decoding vs sampling-based selection ({len(g_ids)} tasks); greedy uses {g_tok:.0f} generated tokens per task")
    display(pd.DataFrame([{"method": k, "accuracy % [95% CI]": fmt_ci(v, idx)} for k, v in ref.items()]))
    g = ref["greedy decoding (1 sample)"]
    for name, other in [(f"semantic@{N_SAMPLES}", ref[f"semantic@{N_SAMPLES}"]), (f"valid_filter@{N_SAMPLES}", ref[f"valid_filter@{N_SAMPLES}"]),
                        ("sampled pass@1", ref["sampled pass@1 (T=%.1f)" % TEMPERATURE])]:
        m, lo, hi, p = paired_diff(other, g, idx); print(f"  {name} - greedy: {100*m:+.1f} points [{100*lo:+.1f}, {100*hi:+.1f}], " + ("p<0.001" if p < 0.001 else f"p={p:.3f}"))

## Step 14 (optional, GPU + CPU): fingerprint inputs that do not come from the tests
The main experiment fingerprints candidates on the call inputs recorded from the benchmark's own tests. Here the **same model writes the example calls** from the function signature and docstring alone (no test, no canonical solution), all 32 candidates are re-executed on those inputs, and the clustering is redone. If `semantic_llm` is close to `semantic`, the main result does not depend on test-derived inputs. Set `RUN_LLM_INPUTS = True` to run it (about 5 minutes of GPU plus 1-2 hours of CPU execution on a free Colab).

In [ ]:
import ast
def sig_ok(prompt, entry, call_src):
    """Does a generated call fit the signature shown in the prompt? (deployable: uses no test and no reference solution)"""
    try:
        fn = next(n for n in ast.walk(ast.parse(prompt + "\n    pass\n"))
                  if isinstance(n, ast.FunctionDef) and n.name == entry)
        call = ast.parse(call_src, mode="eval").body
    except Exception:
        return True
    a = fn.args
    npos = len(a.posonlyargs) + len(a.args); nreq = npos - len(a.defaults)
    names = {x.arg for x in a.posonlyargs + a.args + a.kwonlyargs}
    req_kw = {x.arg for x, d in zip(a.kwonlyargs, a.kw_defaults) if d is None}
    if any(isinstance(x, ast.Starred) for x in call.args) or any(k.arg is None for k in call.keywords):
        return True
    kws = {k.arg for k in call.keywords}
    if not a.vararg and len(call.args) > npos: return False
    if not a.kwarg and not kws <= names: return False
    if len(call.args) + len([k for k in kws if k in names]) < nreq: return False
    return req_kw <= kws

def inputs_prompt(t):
    """Ask the SAME model to write example calls. Only the function signature/docstring is shown: no test, no canonical solution."""
    e = t["entry_point"]; lines = t["prompt"].rstrip().splitlines()
    imports = [l for l in lines if re.match(r"(from|import)\s", l)]
    return ("\n".join(imports) + "\n\n# Function under test (signature and docstring):\n" + "\n".join("# " + l for l in lines)
            + f"\n\n# Three different example calls of `{e}` with realistic arguments, one per line:\nexample_1 = {e}(")

BAD_CALL = re.compile(r"__|\bopen\b|\bexec\b|\beval\b|\bimport\b|\bos\b|\bsys\b|subprocess|\binput\b|\bcompile\b")
def parse_calls(entry, continuations, k=None):
    """Keep up to k distinct, syntactically valid single calls `entry(...)` from the model's continuations."""
    k = k or N_LLM_INPUTS; seen, out = set(), []
    for cont in continuations:
        line = cont.split("\n")[0].strip()
        if not line: continue
        src = f"{entry}({line}"
        try: node = ast.parse(src, mode="eval").body
        except SyntaxError: continue
        if not (isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == entry): continue
        if BAD_CALL.search(src) or src in seen: continue
        seen.add(src); out.append(src)
        if len(out) == k: break
    return out

if RUN_LLM_INPUTS:
    INPUTS_CFG = prepare_generation("llm_inputs_raw", [{"task_id": t, "prompt": inputs_prompt(valid_tasks[t]), "seed": SEED + 1000 + i}
                                                       for i, t in enumerate(run_ids)],
                                    n_samples=8, temperature=0.8, max_new_tokens=96, stops=["\n"], trim=False)
    print("config written:", INPUTS_CFG)

In [ ]:
if RUN_LLM_INPUTS:
    !python gen.py {INPUTS_CFG}

In [ ]:
if RUN_LLM_INPUTS:
    raw = {}
    for line in open(f"{OUT_DIR}/llm_inputs_raw.jsonl"): d = json.loads(line); raw[d["task_id"]] = d["completions"]
    llm_calls, n_fallback = {}, 0
    for tid in run_ids:
        if tid not in raw: continue
        e = valid_tasks[tid]["entry_point"]; calls = [c for c in parse_calls(e, raw[tid], k=2 * N_LLM_INPUTS) if sig_ok(valid_tasks[tid]["prompt"], e, c)][:N_LLM_INPUTS]
        if not calls: calls, n_fallback = [f"{e}()"], n_fallback + 1          # nothing usable: fall back to the no-argument call
        llm_calls[tid] = calls
    print(f"{len(llm_calls)} tasks; {len(llm_calls) - n_fallback} got >=1 usable example call, {n_fallback} fell back to a no-argument call")
    json.dump(llm_calls, open(f"{OUT_DIR}/llm_calls.json", "w"), indent=1)

    LLM_EXEC = f"{OUT_DIR}/llm_exec.pkl"; NOCHECK = "def check(candidate):\n    pass\n"
    cache_l = pickle.load(open(LLM_EXEC, "rb")) if os.path.exists(LLM_EXEC) else {}
    for k, tid in enumerate(tqdm(list(llm_calls), desc="executing (LLM-written inputs)")):
        if tid in cache_l: continue
        t = valid_tasks[tid]
        jobs = [dict(mode="candidate", program=build_program(t["prompt"], c), test=NOCHECK, entry_point=t["entry_point"],
                     inputs=[], input_src=llm_calls[tid], skip_check=True) for c in samples[tid]["completions"][:N_SAMPLES]]
        cache_l[tid] = run_jobs(jobs, progress=False)
        if k % 10 == 0: _save_atomic(cache_l, LLM_EXEC)
    _save_atomic(cache_l, LLM_EXEC)
    for tid, results in cache_l.items():
        sigs = [r["sig"] if r.get("ok") else None for r in results]; v = valid_mask(sigs)
        stats[tid]["valid_llm"], stats[tid]["E_llm"] = v, equiv_matrix(sigs, v)

In [ ]:
if RUN_LLM_INPUTS:
    ids_llm = [t for t in all_ids if "E_llm" in stats[t]]
    pt_llm = fixed_pertask(ids_llm, ns=sorted({n for n in (8, 16, N_SAMPLES) if n <= N_SAMPLES}), views=("test", "llm"))
    print(f"Fingerprint inputs taken from the tests vs written by the LLM ({len(ids_llm)} tasks), accuracy % [95% CI]")
    display(ci_table(pt_llm, ["random", "valid_filter", "semantic", "valid_filter_llm", "semantic_llm", "oracle"]))
    display(diff_table(pt_llm, [("semantic", "semantic_llm"), ("semantic_llm", "valid_filter_llm"), ("semantic_llm", "random")]))

In [ ]:
# LLM-written inputs: no-argument tasks vs tasks with arguments
if RUN_LLM_INPUTS:
    na = set(noarg_ids)
    for name, ids in (("no-argument", [t for t in ids_llm if t in na]),
                      ("with arguments", [t for t in ids_llm if t not in na])):
        pt = fixed_pertask(ids, ns=[32], views=("test", "llm"))
        print(name, len(ids))
        display(ci_table(pt, ["random", "valid_filter", "semantic", "valid_filter_llm", "semantic_llm", "oracle"]))


## Step 15: reproducibility report, and the figure and numbers for the paper
Saves versions, configuration, the list of excluded tasks, the main figure (`fig_main.pdf`) and all numbers (`paper_numbers.json`) into `OUT_DIR`.

In [ ]:
import platform, importlib.metadata as md
def _ver(p):
    try: return md.version(p)
    except Exception: return None
repro = dict(python=platform.python_version(), **{p: _ver(p) for p in ["qiskit", "qiskit-aer", "qiskit-ibm-runtime", "transformers", "torch", "numpy", "pandas"]},
             model_id=MODEL_ID, n_samples=N_SAMPLES, temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
             exec_timeout_s=EXEC_TIMEOUT, circuit_eq=CIRCUIT_EQ, dist_tol=DIST_TOL, unitary_qubit_cap=8, bootstrap_resamples=BOOT_B,
             tasks_in_dataset=len(tasks), tasks_usable=len(valid_tasks), tasks_sampled=len(all_ids), tasks_informative=len(strong_ids),
             tasks_no_argument=len(noarg_ids) if "noarg_ids" in globals() else None)
json.dump(repro, open(f"{OUT_DIR}/repro.json", "w"), indent=1)
print(json.dumps(repro, indent=1))
if "excluded" in globals() and excluded:
    df_ex = pd.DataFrame([{"task_id": k, **v} for k, v in excluded.items()])
    df_ex.to_csv(f"{OUT_DIR}/excluded_tasks.csv", index=False)
    print(f"\n{len(df_ex)} tasks were excluded because their canonical solution fails in this environment (list saved to excluded_tasks.csv):")
    display(df_ex.groupby(["stage", "error"]).size().sort_values(ascending=False).head(10).rename("tasks").reset_index())

In [ ]:
P = pt_syn if "pt_syn" in globals() else pt_all
idx = boot_idx(len(P["ids"])); ns = P["ns"]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
ax = axes[0]
curves = [("random", "tab:gray", "o", "-"), ("valid_filter", "tab:blue", "s", "-"), ("semantic", "tab:red", "D", "-"), ("oracle", "black", "^", "--")]
if "semantic_syntactic" in P["vals"]: curves.insert(3, ("semantic_syntactic", "tab:orange", "v", ":"))
for key, col, mk, ls in curves:
    v = P["vals"][key]; bs = np.array([boot_mean(v[:, j], idx) for j in range(len(ns))]); lo, hi = np.percentile(bs, [2.5, 97.5], axis=1)
    ax.plot(ns, 100 * v.mean(axis=0), marker=mk, color=col, ls=ls, label=key.replace("_", " "))
    if key in ("valid_filter", "semantic"): ax.fill_between(ns, 100 * lo, 100 * hi, color=col, alpha=.15, lw=0)   # bands only where they stay readable
ax.set_xscale("log", base=2); ax.set_xticks(ns); ax.set_xticklabels(ns); ax.set_xlabel("samples per problem (N)")
ax.set_ylabel("accuracy (%)"); ax.set_title(f"All usable tasks (n={len(P['ids'])}); bands: 95% CI over tasks"); ax.grid(alpha=.3); ax.legend(fontsize=8)

ax = axes[1]; v = P["vals"]["semantic"]
ax.plot(P["tokens"].mean(axis=0), 100 * v.mean(axis=0), "D-", color="tab:red", label="semantic, fixed N")
ax.plot(P["tokens"].mean(axis=0), 100 * P["vals"]["random"].mean(axis=0), "o-", color="tab:gray", label="random (pass@1)")
if "ad_tau" in globals():
    ax.plot([r["tokens"].mean() for r in ad_tau.values()], [100 * r["acc"].mean() for r in ad_tau.values()], "*-", color="tab:green", ms=11, label="adaptive, tau rule")
    ax.plot([r["tokens"].mean() for r in ad_beta.values()], [100 * r["acc"].mean() for r in ad_beta.values()], "^-", color="tab:purple", ms=7, label="adaptive, Beta rule (AC)")
if RUN_GREEDY and "ref" in globals():
    ax.plot([g_tok], [100 * ref["greedy decoding (1 sample)"].mean()], "s", color="tab:blue", ms=9, label="greedy decoding")
ax.set_xscale("log"); ax.set_xlabel("avg generated tokens per problem"); ax.set_ylabel("accuracy (%)"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_main.pdf"); plt.savefig(f"{OUT_DIR}/fig_main.png", dpi=200); plt.show()

# every number the paper's tables are built from, in one file
paper = dict(curve_all=curve_all.round(4).to_dict("list"), curve_strong=curve_strong.round(4).to_dict("list"),
             adaptive_all=adapt_all.round(4).to_dict("list"), adaptive_strong=adapt_strong.round(4).to_dict("list"),
             ci_all=ci_table(pt_all).to_dict("list"), gap=gap_stats(pt_all), repro=repro)
json.dump(paper, open(f"{OUT_DIR}/paper_numbers.json", "w"), indent=1, default=str)
print("saved: fig_main.pdf/png, paper_numbers.json and the CSV tables in", OUT_DIR)

## Optional: determinism self-test

In [ ]:
# OPTIONAL self-test: the fingerprints of the most random tasks must be identical across two fresh executions.
samples_chk = {}
for line in open(f"{OUT_DIR}/samples.jsonl"):
    d = json.loads(line); samples_chk[d["task_id"]] = d["completions"]
for n in (31, 48, 142, 143):
    tid = next(t for t in valid_tasks if str(t).endswith(f"/{n}")); t = valid_tasks[tid]
    jobs2 = [dict(mode="candidate", program=build_program(t["prompt"], c), test=t["test"],
                  entry_point=t["entry_point"], inputs=t["inputs"], skip_check=True) for c in samples_chk[tid][:N_SAMPLES]]
    S = [[repr(x["sig"]) if x.get("ok") else None for x in run_jobs(jobs2, progress=False)] for _ in range(2)]
    print(tid, "signatures that differ between two fresh runs (must be 0):", sum(a != b for a, b in zip(*S)))


## What to write in the paper
A LaTeX draft that uses these outputs is provided next to this notebook (`paper/main.tex`). Fill its `\todo{}` boxes from the Step 10-14 outputs and `paper_numbers.json`.
* **Method**: model, N, temperature, fingerprint definition, the assumptions listed at the top, the strategies, the stopping rules.
* **Results**: `fig_main.pdf`; the accuracy table with confidence intervals (10a); the gain decomposition (10a/10b); by difficulty (10c); baselines and ablations (11); adaptive stopping at matched budgets and with a cross-validated threshold (12); greedy decoding (13).
* **Discussion**: the gap between `semantic` and `oracle` (what better verification could still buy), the gap between `semantic` and `valid_filter` (what clustering adds beyond "does it run"), the "correct but different" and "matches canonical but fails" counts.
* **Limitations**: one small model, one benchmark (possible training-data contamination), property-based tests, usable-task and informative-subset selection use the canonical solution, Qiskit version differs from the one the benchmark was written for, unitary comparison capped at 8 qubits.

**Easy extensions if you have time**: repeat with other model sizes or families (change `MODEL_ID` and `OUT_DIR`), a different temperature, or QHE-Hard.